# L10 — Monitoring, drift & injected-fault alerting
**Objective 13.**

**Northfield Grocers context:** Promo weeks change what Northfield customers ask and how they phrase it; a holiday range launch changes the catalog. Week 4 in the lab is a promo week. Monitoring must flag the drifted input feature, stay silent on the stable one, and prove its latency alert fires on an injected fault before the real peak arrives.

**Retail use cases:** Promo-week and holiday drift in customer questions; silent provider model updates raising cost per conversation; index staleness after a range change.

**Platform:** Both platforms. Drift maths and alert logic run anywhere; managed monitors configured per lab guide.

**Done means:** PSI flags the planted drifted feature (title_len, week 4) and not the stable one (attr_count); the injected latency fault raises the alert.

## Step 1 — Load the four-week window (week 4 is a promo week)

In [1]:
# === Lab environment header (identical in every lab) ===
import os, sys, json, time, math, shutil, re, subprocess, importlib, importlib.util
import numpy as np, pandas as pd

def ensure_packages(pkgs):
    """Install any missing pip packages into THIS Python (same mechanism as %pip on Databricks) and import them.
    Fresh packages are importable immediately — no restart. Only labs that need extras call this (L02, L08)."""
    missing = [p for p in pkgs if importlib.util.find_spec(p.replace("-", "_")) is None]
    if not missing:
        print("Packages present:", pkgs); return
    print("Installing missing packages into", sys.executable, ":", missing)
    cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        r = subprocess.run(cmd + ["--break-system-packages"], capture_output=True, text=True)   # local system Pythons
    if r.returncode != 0:
        raise ImportError("pip could not install " + str(missing) + ". Ask the admin to add them as cluster libraries "
                          "(Compute → Libraries → PyPI) or use an internal index. pip said: " + r.stderr[-600:])
    importlib.invalidate_caches()
    for p in missing: importlib.import_module(p.replace("-", "_"))
    print("Installed and imported:", missing)

# Mode: "GPU" runs the full lab on Azure GPU compute; "SMOKE" runs the CPU/synthetic path anywhere.
LAB_MODE = os.environ.get("LAB_MODE") or ("GPU" if shutil.which("nvidia-smi") else "SMOKE")

# Data folder: env override → package-relative (../../data) → Unity Catalog volume → search the workspace once
_candidates = [os.environ.get("DATA_DIR"), os.path.abspath(os.path.join(os.getcwd(), "..", "..", "data")), "/Volumes/northfield/llmops/labdata"]
DATA_DIR = next((c for c in _candidates if c and os.path.exists(os.path.join(c, "catalog_items.csv"))), None)
if DATA_DIR is None:
    import glob
    _hits = [h for root in ("/Workspace", "/Volumes", os.path.expanduser("~")) if os.path.isdir(root)
             for h in glob.glob(os.path.join(root, "**", "catalog_items.csv"), recursive=True)][:1]
    DATA_DIR = os.path.dirname(_hits[0]) if _hits else None
if DATA_DIR is None:
    raise FileNotFoundError("Lab data not found. Upload the package's data/ folder to a Unity Catalog volume and set "
                            "os.environ['DATA_DIR'] = '/Volumes/<catalog>/<schema>/<volume>' in a cell above this one.")
def gpu_only(msg):
    """Called wherever a step needs a GPU / model download that the smoke path cannot run."""
    print(f"[{LAB_MODE}] GPU-only step not executed here: {msg}")
def check(cond, msg):
    """Binary 'done means' assertion — prints PASS/FAIL and raises on FAIL so the notebook stops."""
    print(("PASS " if cond else "FAIL ") + msg); assert cond, msg
print(f"LAB_MODE={LAB_MODE}  DATA_DIR={DATA_DIR}  python={sys.version.split()[0]}")

LAB_MODE=SMOKE  DATA_DIR=/home/claude/llmops_labs/data  python=3.12.3


In [2]:
mon = pd.read_csv(os.path.join(DATA_DIR, "monitoring_window.csv"))
print(mon.groupby("week")[["title_len", "attr_count", "latency_ms"]].mean().round(2))

      title_len  attr_count  latency_ms
week                                   
1         41.54        2.99      117.40
2         41.59        3.01      120.47
3         41.73        3.01      121.19
4         57.98        2.98      117.81


## Step 2 — Population Stability Index by hand
*Why:* PSI is what most monitoring products report; know its definition (Σ (p−q)·ln(p/q) over bins) and its usual thresholds (<0.1 stable, 0.1–0.25 watch, >0.25 drift).

In [3]:
def psi(ref, cur, bins=10):
    edges = np.quantile(ref, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = -np.inf, np.inf
    p = np.histogram(ref, edges)[0] / len(ref) + 1e-6; q = np.histogram(cur, edges)[0] / len(cur) + 1e-6
    return float(np.sum((p - q) * np.log(p / q)))

ref = mon[mon.week == 1]
for f in ("title_len", "attr_count"):
    print(f, {w: round(psi(ref[f], mon[mon.week == w][f]), 3) for w in (2, 3, 4)})
check(psi(ref.title_len, mon[mon.week == 4].title_len) > 0.25 and psi(ref.attr_count, mon[mon.week == 4].attr_count) < 0.1, "PSI flags title_len week 4 only")

title_len {2: 0.028, 3: 0.073, 4: 6.985}
attr_count {2: 0.018, 3: 0.035, 4: 0.021}
PASS PSI flags title_len week 4 only


## Step 3 — Cross-check with a KS test
*Why:* PSI depends on binning; the two-sample KS statistic does not. Two methods agreeing is stronger evidence than one.

In [4]:
from scipy.stats import ks_2samp
ks = {f: ks_2samp(ref[f], mon[mon.week == 4][f]).pvalue for f in ("title_len", "attr_count")}
print("KS p-values week 4 vs week 1:", {k: f"{v:.2e}" for k, v in ks.items()})
check(ks["title_len"] < 0.01 < ks["attr_count"], "KS agrees with PSI")

KS p-values week 4 vs week 1: {'title_len': '8.80e-189', 'attr_count': '9.99e-01'}
PASS KS agrees with PSI


## Step 4 — Latency and cost-per-request, then inject a fault
*Why:* an alert you have never seen fire is not an alert. Compute p95 latency and cost per request per week, then inject a +40 % latency fault into a copy of week 4 and confirm the rule triggers.

In [5]:
usd_per_gpu_hr, req_per_hr = 3.673, 20000   # A100 SKU price (approx, from content.py) and an assumed load
def metrics(df):
    return dict(p95_ms=float(np.percentile(df.latency_ms, 95)), cost_per_req_usd=usd_per_gpu_hr / req_per_hr)
weekly = {w: metrics(mon[mon.week == w]) for w in range(1, 5)}
baseline_p95 = np.mean([weekly[w]["p95_ms"] for w in (1, 2, 3)])
def alert(p95, baseline, threshold=1.25): return p95 > baseline * threshold
faulty = mon[mon.week == 4].copy(); faulty["latency_ms"] *= 1.4
print("week4 p95", round(weekly[4]["p95_ms"], 1), "| baseline", round(baseline_p95, 1), "| faulty p95", round(metrics(faulty)["p95_ms"], 1))
check(not alert(weekly[4]["p95_ms"], baseline_p95) and alert(metrics(faulty)["p95_ms"], baseline_p95), "alert silent on normal week, fires on injected fault")

week4 p95 234.4 | baseline 223.3 | faulty p95 328.2
PASS alert silent on normal week, fires on injected fault


## Step 5 — Configure the managed monitor (platform)
*Why:* the hand-rolled version explains the numbers; the managed one runs unattended. Databricks: `Lakehouse Monitoring` on an inference table with a baseline table = week 1. Azure ML: model monitor with data-drift signal on the deployment's collected data. Both are configured through the UI/SDK per lab guide §Step 5 — API names are version-sensitive.

In [6]:
if LAB_MODE == "GPU":
    gpu_only("Create the monitor per lab guide §Step 5 and screenshot the drift chart for the deliverable.")
else:
    print("Local: drift maths and alert logic verified.")
print("L10 complete.")

Local: drift maths and alert logic verified.
L10 complete.
